# Data cleaning

## Load data

In [ ]:
# Load CSV file
import pandas as pd
df = pd.read_csv("45849-s3-sf_TEXT.csv")

# Remove question-text row and ImportId row
df = df.iloc[2:].copy()

# Create working dataset
data = df.copy()

print("Starting sample:", len(data))

## Check variables

In [ ]:
print(data.columns.tolist())

### Remove irrelevant variables

In [ ]:
data = data.drop(columns=[
    'StartDate', 'EndDate', 'Status', 'IPAddress', 'Progress', 'Duration (in seconds)', 'RecordedDate', 'ResponseId', 'RecipientLastName', 'RecipientFirstName', 'RecipientEmail', 
    'ExternalReference', 'LocationLatitude', 'LocationLongitude', 'DistributionChannel', 'UserLanguage', 'Q_RecaptchaScore', 'Gender_4_TEXT', 'Founder Stage', 'Projects', 
    'Uncertainty', 'Mini-IPIP 1', 'Mini-IPIP 2', 'Mini-IPIP 3', 'Mini-IPIP 4', 'IPIP 120 1', 'IPIP 120 2', 'IPIP 120 3', 'IPIP 120 4', 'APS 1', 'APS 2', 'APS 3', 'APS 4', 
    'APS 5', 'APS 6', 'APS 7', 'APS 8', 'APS 9', 'APS 10', 'APS 11', 'APS 12', 'APS 13', 'APS 14', 'APS 15', 'APS 16', 'APS 17', 'APS 18', 'APS 19', 'APS 20', 'clicked', 
    'norms', 'project', 'source', 'results', 'Q_DataPolicyViolations', 'CPM 2', 'CPM 3', 'CPM 5', 'CPM 6', 'CPM 8', 'CPM 9', 'CPM 11', 'CPM 12', 'CPM 14', 'CPM 15', 'CPM 17', 
    'CPM 18', 'CPM 20', 'CPM 21', 'CPM 23', 'CPM 24', 'CPM 26', 'CPM 27'
])
print(data.columns.tolist())

### Change text to numbers

In [ ]:
# CPM questionnaire items
cpm_items = [
    "CPM 1",
    "CPM 4",
    "CPM 7",
    "CPM 10",
    "CPM 13",
    "CPM 16",
    "CPM 19",
    "CPM 22",
    "CPM 25"
]

# PtP questionnaire items
ptp_original = [f"PtP {i}" for i in range(1, 65)]


In [ ]:
cpm_mapping = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Neither agree nor disagree": 3,
    "Agree": 4,
    "Strongly agree": 5
}
data[cpm_items] = (
    data[cpm_items]
    .replace(cpm_mapping)
)

data[cpm_items] = data[cpm_items].astype("Int64")

In [ ]:
ptp_mapping = {
    "Never or hardly ever": 1,
    "Rarely": 2,
    "Sometimes": 3,
    "About half the time": 4,
    "Often": 5,
    "Very often": 6,
    "Always or nearly always": 7
}
data[ptp_original] = (
    data[ptp_original]
    .replace(ptp_mapping)
)

data[ptp_original] = data[ptp_original].astype("Int64")

In [ ]:
print(data[cpm_items].head())
print(data[ptp_original].head())

In [ ]:
# Check text vs numbers
print(data.dtypes)

In [ ]:
# Columns that should contain whole numbers
numeric_columns = [
    "Age",
    "Time",
    "Months",
    "Years",
]

# Convert columns to numeric first
for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Convert to whole-number format while allowing missing values
data[numeric_columns] = data[numeric_columns].astype("Int64")

In [ ]:
print(data.dtypes)

### Check data

In [ ]:
data[cpm_items + ptp_original].head()

### Check data again

In [ ]:
# Count answered questions for each participant
data["CPM_completed"] = data[cpm_items].notna().sum(axis=1)
data["PtP_completed"] = data[ptp_original].notna().sum(axis=1)

print("CPM: at least 1 question answered =", (data["CPM_completed"] > 0).sum())
print("PtP: at least 1 question answered =", (data["PtP_completed"] > 0).sum())

## Exclusion criteria

In [ ]:
# Check data
print(data.head())
print(data.shape)

# Check age
print("Missing age:")
print(data["Age"].isna().sum())

print("\nAge distribution:")
print(data["Age"].describe())

# Check founder responses
print("\nFounder responses:")
print(data["Founder"].value_counts(dropna=False))

### Apply exclusion criteria (age, founder)

In [ ]:
# 1. Remove missing age
data = data[data["Age"].notna()]
print("After missing age removed:", len(data))

# 2. Remove participants under 18
data = data[data["Age"] >= 18]
print("After under 18 removed:", len(data))

# 3. Remove missing founder response
data = data[data["Founder"].notna()]
print("After missing founder response removed:", len(data))

# 4. Remove non-founders
data = data[data["Founder"] != "No"]
print("After non-founders removed:", len(data))


In [ ]:
# Count completed items
data["CPM_completed"] = data[cpm_items].notna().sum(axis=1)
data["PtP_completed"] = data[ptp_original].notna().sum(axis=1)


# Remove participants who didn't start either inventory
didnt_start_either = (
    (data["CPM_completed"] == 0) &
    (data["PtP_completed"] == 0)
)

print(
    "Didn't start either inventory:",
    didnt_start_either.sum()
)

data = data[~didnt_start_either].copy()

print(
    "After removing participants who didn't start either inventory:",
    len(data)
)


# Remove participants who completed only one inventory
only_one_inventory = (
    ((data["CPM_completed"] > 0) & (data["PtP_completed"] == 0)) |
    ((data["CPM_completed"] == 0) & (data["PtP_completed"] > 0))
)

print(
    "Only completed one inventory:",
    only_one_inventory.sum()
)

data = data[~only_one_inventory].copy()

print("Final sample:", len(data))

## Reverse scoring

In [ ]:
# Select all 64 PtP items from the main dataset
ptp_original = data[[f"PtP {i}" for i in range(1, 65)]].copy()

print(type(ptp_original))

In [ ]:
# Define items that need reverse scoring
reverse_ptp_original = [
    "PtP 5", "PtP 6", "PtP 7", "PtP 8",
    "PtP 11",
    "PtP 13", "PtP 14", "PtP 15", "PtP 16", "PtP 17",
    "PtP 18", "PtP 19",
    "PtP 24", "PtP 25",
    "PtP 30", "PtP 31", "PtP 32",
    "PtP 34",
    "PtP 37",
    "PtP 39", "PtP 40", "PtP 41", "PtP 42", "PtP 43", "PtP 44",
    "PtP 48", "PtP 49", "PtP 50", "PtP 51", "PtP 52",
    "PtP 56", "PtP 57", "PtP 58",
    "PtP 61"
]
# Make a copy of the original PtP data
ptp_items = ptp_original.copy()

# Reverse score the selected items
ptp_items[reverse_ptp_original] = 8 - ptp_original[reverse_ptp_original]

In [ ]:
print(ptp_items.head())

## Creating the moderator variable

In [ ]:
# Define the R&D classifications
industry_classification = {
    "Agriculture": "lRD",
    "Forestry": "lRD",
    "Mining": "lRD",
    "Energy and water supply": "lRD",
    "Waste/Recycling": "lRD",
    "Construction": "lRD",
    "Trades e.g. Plumbing/Electrical": "lRD",
    "Manufacturing": "hRD",
    "Wholesale Trade": "lRD",
    "Retail Trade": "lRD",
    "Real Estate": "lRD",
    "Hospitality": "lRD",
    "Delivery Services": "lRD",
    "Transport/Logistics": "lRD",
    "Cleaning/Maintenance": "lRD",
    "Technical services": "hRD",
    "Travel/Tourism": "lRD",
    "Entertainment": "lRD",
    "Recreation": "lRD",
    "Arts": "lRD",
    "Education/Training": "hRD",
    "Research/Academia": "hRD",
    "Health care": "hRD",
    "Social Services": "hRD",
    "Administration": "lRD",
    "IT/Telecommunication": "hRD",
    "Finance/Insurance": "lRD",
    "Environment/Sustainability": "lRD"
}

In [ ]:
# Split multiple industries for each participant
data["Industry_list"] = data["Industry.1"].str.split(",")

#Classify each selected industry
data["RD_categories"] = data["Industry_list"].apply(
    lambda industries: [
        industry_classification.get(industry.strip())
        for industry in industries
    ] if isinstance(industries, list) else []
)

# Count low and high R&D industries
data["lRD_count"] = data["RD_categories"].apply(
    lambda x: x.count("lRD")
)

data["mRD_count"] = data["RD_categories"].apply(
    lambda x: x.count("mRD")
)

data["hRD_count"] = data["RD_categories"].apply(
    lambda x: x.count("hRD")
)

# Classifications
def classify_rd(row):
    counts = {
        "lRD": row["lRD_count"],
        "mRD": row["mRD_count"],
        "hRD": row["hRD_count"]
    }

    # No valid industry information
    if sum(counts.values()) == 0:
        return pd.NA

    # Find the maximum number of industries
    max_count = max(counts.values())

    # Find categories with the maximum count
    tied = [
        category for category, count in counts.items()
        if count == max_count
    ]

    # If only one category has the highest count
    if len(tied) == 1:
        return tied[0]

    # Tie-breaking rules
    # lRD + mRD → mRD
    # lRD + hRD → mRD
    # mRD + hRD → hRD

    if "hRD" in tied and "mRD" in tied:
        return "hRD"

    if "mRD" in tied and "lRD" in tied:
        return "mRD"

    if "hRD" in tied and "lRD" in tied:
        return "mRD"

    return pd.NA

In [ ]:
data["RD_category"] = data.apply(classify_rd, axis=1)

In [ ]:
# Check final groups
print(data["RD_category"].value_counts(dropna=False))

In [ ]:
print(
    data.loc[
        data["RD_category"] == "mRD",
        [
            "Industry.1",
            "RD_categories",
            "lRD_count",
            "mRD_count",
            "hRD_count"
        ]
    ].to_string(index=False)
)